# 解答① NumPy で線形回帰・勾配降下を手実装

> **このファイルは `exercises/ex_01_linear_regression.ipynb` の解答です。**  
> 課題に取り組む前に見ないようにしてください。

---

## セットアップ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)
print('セットアップ完了')

---

## Step 1: データの生成

In [ ]:
N = 50
x = np.linspace(0, 10, N)
y_true = 2 * x + 1
y = y_true + np.random.randn(N) * 2

print(f'データ数: {N}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, color='#57534E', s=25, alpha=0.7, label='観測値（ノイズあり）')
ax.plot(x, y_true, color='#0D4A38', linewidth=1.5, linestyle='--', label='真のモデル (y = 2x + 1)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_title('生成データの散布図')
ax.legend()
plt.tight_layout()
plt.show()

---

## Step 2: モデルの定義（解答）

線形回帰モデル $\hat{y} = wx + b$ を実装します。

In [ ]:
def predict(x, w, b):
    """線形回帰モデル: ŷ = wx + b"""
    return w * x + b    # ← 解答: 1行で実装

# 確認
assert predict(np.array([1.0, 2.0]), 2.0, 1.0).tolist() == [3.0, 5.0], '計算が合いません'
print('predict() ✓')

---

## Step 3: 損失関数（MSE）の実装（解答）

$$L = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2$$

In [ ]:
def mse_loss(y_true, y_pred):
    """平均二乗誤差（MSE）を計算する"""
    return np.mean((y_true - y_pred) ** 2)    # ← 解答: 1行で実装

# 確認
assert abs(mse_loss(np.array([1.0, 2.0]), np.array([1.5, 1.5])) - 0.25) < 1e-6, '計算が合いません'
print('mse_loss() ✓')

loss_init = mse_loss(y, predict(x, w=0.0, b=0.0))
loss_true = mse_loss(y, predict(x, w=2.0, b=1.0))
print(f'初期損失 (w=0, b=0):    {loss_init:.4f}')
print(f'真のパラメータの損失:   {loss_true:.4f}')

---

## Step 4: 勾配降下法の実装（解答）

$$\frac{\partial L}{\partial w} = -\frac{2}{N}\sum_i x_i(y_i - \hat{y}_i), \quad \frac{\partial L}{\partial b} = -\frac{2}{N}\sum_i(y_i - \hat{y}_i)$$

$$w \leftarrow w - \alpha \frac{\partial L}{\partial w}, \quad b \leftarrow b - \alpha \frac{\partial L}{\partial b}$$

In [ ]:
alpha    = 0.01
n_epochs = 300
w, b     = 0.0, 0.0
history  = []

for epoch in range(n_epochs):
    y_pred = predict(x, w, b)
    loss   = mse_loss(y, y_pred)
    history.append(loss)

    # 解答: 勾配の計算
    dw = -2 / N * np.sum(x * (y - y_pred))
    db = -2 / N * np.sum(y - y_pred)

    # 解答: パラメータの更新
    w = w - alpha * dw
    b = b - alpha * db

    if epoch % 50 == 0:
        print(f'Epoch {epoch:3d} | loss={loss:.4f} | w={w:.4f} | b={b:.4f}')

print(f'\n最終結果: w={w:.4f}, b={b:.4f}  （真: w=2.0, b=1.0）')

---

## Step 5: 損失曲線と予測結果を可視化

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(history, color='#0D4A38', linewidth=1.8)
ax1.axhline(y=loss_true, color='#A8A29E', linestyle='--', linewidth=1, label=f'理論最小値 = {loss_true:.2f}')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE Loss'); ax1.set_title('損失曲線')
ax1.legend()

ax2.scatter(x, y, color='#57534E', s=25, alpha=0.6)
ax2.plot(x, predict(x, w, b), color='#0D4A38', linewidth=2, label=f'学習後 (w={w:.3f}, b={b:.3f})')
ax2.plot(x, 2*x+1, color='#A8A29E', linestyle='--', linewidth=1.5, label='真 (w=2, b=1)')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_title('学習後の予測')
ax2.legend()

plt.tight_layout()
plt.show()

---

## チャレンジ問題の解答

3種類の学習率で学習を実行し、損失曲線を重ねて描画します。

In [ ]:
alphas = [0.001, 0.01, 0.1]
colors = ['#0D4A38', '#7C5C00', '#991B1B']

fig, ax = plt.subplots(figsize=(8, 4))

for alpha_exp, color in zip(alphas, colors):
    w_exp, b_exp = 0.0, 0.0
    hist_exp = []

    for epoch in range(300):
        y_pred_exp = predict(x, w_exp, b_exp)
        loss_exp = mse_loss(y, y_pred_exp)
        hist_exp.append(min(loss_exp, 500))

        dw = -2 / N * np.sum(x * (y - y_pred_exp))
        db = -2 / N * np.sum(y - y_pred_exp)
        w_exp -= alpha_exp * dw
        b_exp -= alpha_exp * db

    ax.plot(hist_exp, color=color, linewidth=1.8, label=f'alpha={alpha_exp}')

ax.axhline(y=loss_true, color='#A8A29E', linestyle='--', linewidth=1, label='理論最小値')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss'); ax.set_title('学習率による損失曲線の違い')
ax.legend()
plt.tight_layout()
plt.show()